#### Fake News Detection Pipeline using Bi-LSTM and TensorFlow.
Author: Yvan Bonival FEUGANG

In [1]:
# Import all the packages needed for the project and define the configuration variables

import io
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Configuration générale
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42
MAX_VOCAB = 10_000
MAX_LEN = 256
BATCH_SIZE = 64
EPOCHS = 5
EMBEDDING_DIM = 128

In [2]:
# Chargement et Préparation des Données

def load_and_preprocess_data(fake_path: str = "Fake.csv", true_path: str = "True.csv") -> pd.DataFrame:
    fake_df = pd.read_csv(fake_path)
    true_df = pd.read_csv(true_path)

    fake_df["class"] = 0
    true_df["class"] = 1

    df = pd.concat([fake_df, true_df], ignore_index=True)

    # Fusion Titre + Contenu pour un meilleur contexte sémantique
    df["full_text"] = df["title"].fillna("") + " " + df["text"].fillna("")

    # Suppression des colonnes non généralisables
    df = df[["full_text", "class"]].dropna()
    return df


def clean_text(text: str) -> str:
    """Nettoyage regex rapide du texte brut."""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Chargement des données...")
df = load_and_preprocess_data()
df["full_text"] = df["full_text"].apply(clean_text)

# Partitionnement Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"].values,
    df["class"].values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["class"].values,
)

Chargement des données...


In [3]:
# 2. Vectorisation du Texte (Keras)

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorizer.adapt(X_train)

# Création des datasets tf.data
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10_000, seed=RANDOM_STATE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)